# 02 - Feature Engineering (Urban Training Set)

Builds the modelling-ready feature matrix from the filtered urban accident data.
All transformers defined here are applied identically to the rural test set in notebook 04.

| Dataset | Rows | Fatal accidents | Fatality rate |
|---|---|---|---|
| Urban (train) | 3,544 | 64 | 1.81% |

**Steps**
1. Load raw urban data
2. Target variable
3. Speed-limit remapping
4. Road-width binning
5. Time features (`hour_bin`, `is_weekend`)
6. One-hot encoding
7. Drop leakage / low-variance columns
8. Sanity checks & save

**Output:** `urban_features.csv`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load Raw Urban Data

In [3]:
# -- Data paths -- update to your local environment
DATA_DIR   = '../data/'
urban_path = DATA_DIR + 'urban_filter.csv'
data = pd.read_csv(urban_path, parse_dates=['accident_time'])

print(f'Loaded: {data.shape[0]:,} rows, {data.shape[1]} columns')
print(f'Date range: {data["accident_time"].min().date()} -> {data["accident_time"].max().date()}')

Loaded: 3,544 rows, 27 columns
Date range: 2018-01-01 -> 2025-12-30


## 2. Target Variable

In [4]:
data['is_fatal'] = (data['deaths'].fillna(0) > 0).astype(int)

print(data['is_fatal'].value_counts())
print(f'\nFatality rate: {data["is_fatal"].mean()*100:.2f}%')
print(f'Fatal cases:   {data["is_fatal"].sum()}')

is_fatal
0    3480
1      64
Name: count, dtype: int64

Fatality rate: 1.81%
Fatal cases:   64


## 3. Speed-Limit Remapping

Collapse low-n edge values into the nearest meaningful bin.
Valid urban limits after remap: **20 / 30 / 40 / 50 / 70**.

| Original | Remapped | Reason |
|---|---|---|
| 5, 10, 15 | 20 | Near-zero limits, likely data entry |
| 25 | 30 | Non-standard, rounds to 30 |
| 45 | 40 | Non-standard, rounds to 40 |
| 60, 80 | 70 | Rural-style limits rare in urban context |

In [5]:
speed_remap = {5: 20, 10: 20, 15: 20, 25: 30, 45: 40, 60: 70, 80: 70}

data['speed_limit'] = data['speed_limit'].replace(speed_remap)

print('Speed-limit distribution after remap:')
print(data['speed_limit'].value_counts().sort_index())
print('\nFatality rate by speed limit:')
print(
    data.groupby('speed_limit')['is_fatal']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'fatality_rate', 'count': 'n'})
      .round(4)
)

Speed-limit distribution after remap:
speed_limit
20     143
30     333
40     182
50    2724
70     162
Name: count, dtype: int64

Fatality rate by speed limit:
             fatality_rate     n
speed_limit                     
20                  0.0070   143
30                  0.0090   333
40                  0.0220   182
50                  0.0173  2724
70                  0.0556   162


## 4. Road-Width Binning

| Bin | Width (m) | Rationale |
|---|---|---|
| narrow | 3-5 | Single-lane / alley |
| standard | 6-9 | Two-lane urban |
| wide | 10-14 | Multi-lane / boulevard |
| arterial | 15+ | Major urban arterial |

In [6]:
width_bins   = [0, 5, 9, 14, np.inf]
width_labels = ['narrow', 'standard', 'wide', 'arterial']

data['road_width_bin'] = pd.cut(
    data['road_width'],
    bins=width_bins,
    labels=width_labels,
    right=True
)

print('Road-width bin distribution:')
print(data['road_width_bin'].value_counts())
print('\nFatality rate per road-width bin:')
print(
    data.groupby('road_width_bin', observed=True)['is_fatal']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'fatality_rate', 'count': 'n'})
      .round(4)
)

Road-width bin distribution:
road_width_bin
standard    1548
wide         954
narrow       592
arterial     450
Name: count, dtype: int64

Fatality rate per road-width bin:
                fatality_rate     n
road_width_bin                     
narrow                 0.0236   592
standard               0.0161  1548
wide                   0.0210   954
arterial               0.0111   450


## 5. Time Features

- **`hour_bin`**: `night` (22-05) / `rush` (07-09, 15-18) / `evening` (19-21) / `day` (rest)
- **`is_weekend`**: Saturday = 5, Sunday = 6

In [7]:
hour = data['accident_time'].dt.hour
dow  = data['accident_time'].dt.dayofweek

def assign_hour_bin(h):
    if h in range(22, 24) or h in range(0, 6):
        return 'night'
    elif h in list(range(7, 10)) + list(range(15, 19)):
        return 'rush'
    elif h in range(19, 22):
        return 'evening'
    else:
        return 'day'

data['hour_bin']   = hour.map(assign_hour_bin)
data['is_weekend'] = (dow >= 5).astype(int)

print('Hour-bin distribution:')
print(data['hour_bin'].value_counts())
print('\nFatality rate by hour_bin:')
print(
    data.groupby('hour_bin')['is_fatal']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'fatality_rate', 'count': 'n'})
      .sort_values('fatality_rate', ascending=False)
      .round(4)
)
print('\nFatality rate by is_weekend:')
print(
    data.groupby('is_weekend')['is_fatal']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'fatality_rate', 'count': 'n'})
      .round(4)
)

Hour-bin distribution:
hour_bin
rush       1692
day        1022
evening     476
night       354
Name: count, dtype: int64

Fatality rate by hour_bin:
          fatality_rate     n
hour_bin                     
day              0.0254  1022
night            0.0254   354
evening          0.0168   476
rush             0.0124  1692

Fatality rate by is_weekend:
            fatality_rate     n
is_weekend                     
0                  0.0179  2738
1                  0.0186   806


## 6. One-Hot Encoding

`settlement` is included here for the urban set (22 meaningful city-level values).
It is excluded from rural feature engineering in notebook 04 (549 unique values, all map to zero when aligned to urban columns).

In [8]:
ohe_cols = [
    'accident_type',
    'accident_type_detailed',
    'road_condition',
    'lighting',
    'settlement',
    'road_width_bin',
    'hour_bin',
    'weather',
]

df_enc = pd.get_dummies(
    data,
    columns=ohe_cols,
    drop_first=False,
    dtype=int
)

print(f'Shape after OHE: {df_enc.shape}')
ohe_generated = [c for c in df_enc.columns if any(c.startswith(b + '_') for b in ohe_cols)]
print(f'OHE columns generated: {len(ohe_generated)}')

Shape after OHE: (3544, 99)
OHE columns generated: 76


## 7. Drop Leakage / Low-Variance Columns

| Column | Reason for dropping |
|---|---|
| `deaths`, `injured` | Direct leakage -- source of target |
| `road_type_detailed`, `road_surface_condition` | Leakage risk / low variance |
| `accident_scenario` | 65 unique values, many n=1 |
| `join_distance` | Noisy spatial join artefact |
| `road_type` | Subsumed by `road_width_bin` |
| `road_width` | Raw value replaced by bin |
| `x_coord`, `y_coord` | GPS coordinates, no cluster feature |
| `accident_id`, `accident_time` | Identifiers |
| `county`, `municipality` | High-cardinality admin fields |
| `involving_motorvehicle_driver` | Constant after filter |

In [9]:
drop_cols = [
    'deaths', 'injured',
    'road_type_detailed', 'road_surface_condition',
    'accident_scenario',
    'join_distance',
    'road_type',
    'road_width',
    'x_coord', 'y_coord',
    'accident_id', 'accident_time',
    'county', 'municipality',
    'involving_motorvehicle_driver',
]

drop_existing = [c for c in drop_cols if c in df_enc.columns]
df_model = df_enc.drop(columns=drop_existing)

print(f'Dropped {len(drop_existing)} columns.')
print(f'Final feature matrix: {df_model.shape}')

Dropped 15 columns.
Final feature matrix: (3544, 84)


## 8. Sanity Checks & Save

In [10]:
print(f'Shape:         {df_model.shape}')
print(f'Fatality rate: {df_model["is_fatal"].mean()*100:.2f}%')
print(f'Fatal cases:   {df_model["is_fatal"].sum()}')
print(f'Total nulls:   {df_model.isnull().sum().sum()}')
print('\nFeature columns:')
print(df_model.columns.tolist())

Shape:         (3544, 84)
Fatality rate: 1.81%
Fatal cases:   64
Total nulls:   0

Feature columns:
['no_safety_equipment', 'motorcyclist', 'pedestrian', 'elderly_driver', 'underage', 'speed_limit', 'is_fatal', 'is_weekend', 'accident_type_Jalakäijaõnnetus', 'accident_type_Kokkupõrge', 'accident_type_Ühesõidukiõnnetus', 'accident_type_detailed_Kokkupõrge ees liikuva sõidukiga', 'accident_type_detailed_Kokkupõrge ees seisva sõidukiga', 'accident_type_detailed_Kokkupõrge jalakäijaga', 'accident_type_detailed_Kokkupõrge sõidukiga küljelt', 'accident_type_detailed_Kokkupõrge teel oleva takistusega', 'accident_type_detailed_Kokkupõrge teevälise takistusega', 'accident_type_detailed_Kokkupõrge vastutuleva sõidukiga', 'accident_type_detailed_Sõiduki teelt väljasõit', 'accident_type_detailed_Sõiduki ümberpaiskumine teel', 'accident_type_detailed_Sõidukite külgkokkupõrge', 'road_condition_Ajutine liikluskorraldus', 'road_condition_Foor ei tööta', 'road_condition_Foor ei tööta, ei ole nähtav', '

In [11]:
df_model.to_csv('urban_features.csv', index=False)
print('Saved: urban_features.csv')
df_model.head(3)

Saved: urban_features.csv


,no_safety_equipment,motorcyclist,pedestrian,elderly_driver,underage,speed_limit,is_fatal,is_weekend,accident_type_Jalakäijaõnnetus,accident_type_Kokkupõrge,accident_type_Ühesõidukiõnnetus,accident_type_detailed_Kokkupõrge ees liikuva sõidukiga,accident_type_detailed_Kokkupõrge ees seisva sõidukiga,accident_type_detailed_Kokkupõrge jalakäijaga,accident_type_detailed_Kokkupõrge sõidukiga küljelt,accident_type_detailed_Kokkupõrge teel oleva takistusega,accident_type_detailed_Kokkupõrge teevälise takistusega,accident_type_detailed_Kokkupõrge vastutuleva sõidukiga,accident_type_detailed_Sõiduki teelt väljasõit,accident_type_detailed_Sõiduki ümberpaiskumine teel,accident_type_detailed_Sõidukite külgkokkupõrge,road_condition_Ajutine liikluskorraldus,road_condition_Foor ei tööta,"road_condition_Foor ei tööta, ei ole nähtav","road_condition_Foor ei tööta, ei ole nähtav,Puu (post) teepeenral (eraldusribal),Sõidutee on korras",...,settlement_Põhja-Tallinna linnaosa,settlement_Rakvere linn,settlement_Sillamäe linn,settlement_Tartu linn,settlement_Valga linn,settlement_Viljandi linn,settlement_Võru linn,road_width_bin_narrow,road_width_bin_standard,road_width_bin_wide,road_width_bin_arterial,hour_bin_day,hour_bin_evening,hour_bin_night,hour_bin_rush,weather_Lumised olud,weather_Madal vastu paistev päike,weather_Muu,"weather_Pilvine,Tugev tuul",weather_Pilvised olud,weather_Selged olud,weather_Tuisune või tormine,weather_Udused olud,weather_Unknown,weather_Vihmasadu
0,False,False,True,True,True,50,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,True,False,True,False,False,30,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0
2,True,False,True,False,False,50,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0


---

## Summary

**Transformers defined here, applied identically to rural in notebook 04:**
- Speed-limit remap: `{5->20, 10->20, 15->20, 25->30, 45->40, 60->70, 80->70}`
- Road-width bins: narrow (3-5m) / standard (6-9m) / wide (10-14m) / arterial (15m+)
- Hour bins: night (22-05) / rush (07-09, 15-18) / evening (19-21) / day
- OHE columns: accident_type, accident_type_detailed, road_condition, lighting, settlement, road_width_bin, hour_bin, weather

**Dropped:** `deaths`, `injured`, `road_type_detailed`, `road_surface_condition`, `accident_scenario`, `join_distance`, `road_type`, `road_width`, `x_coord`, `y_coord`, `accident_id`, `accident_time`, `county`, `municipality`, `involving_motorvehicle_driver`

**Output:** `urban_features.csv` (3,544 rows, 64 fatal, 1.81% fatality rate)

**Next -> `03_modelling.ipynb`:** train/val split, Logistic Regression, Random Forest, threshold calibration.